# 📊 P5: Análise de Inadimplência — Condomínio Humaitá

**Objetivo**: Rastrear `REC.MULTA+C.M.+JRS.` por apartamento, identificar inadimplentes recorrentes e calcular % de cobrança vs total devido.

**Período**: mai/2022–jun/2026  
**Fontes**: Prestações (mai/2022–ago/2023) + Extratos (ago/2025–jun/2026)  
**Output**: `inadimplentes.csv`, `inadimplencia_por_mes.csv`, gráficos

---

## 1. Imports e Configuração

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings("ignore")

# Configurar estilos
plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")

# Diretórios
CSV_DIR = Path("..") / "exports" / "csv"
FIG_DIR = Path("..") / "exports" / "figs"
FIG_DIR.mkdir(exist_ok=True)

print(f"✓ CSV_DIR: {CSV_DIR}")
print(f"✓ FIG_DIR: {FIG_DIR}")

## 2. Carregar Dados (Prestações + Extratos)

In [ ]:
print("\n" + "="*80)
print("SEÇÃO 1: Carregando dados")
print("="*80)

# Carregar prestações
df_prest = pd.read_csv(CSV_DIR / "prestacoes.csv")
print(f"✓ Prestações carregadas: {len(df_prest)} registros (mai/2022–ago/2023)")

# Carregar extratos
df_extra = pd.read_csv(CSV_DIR / "extratos.csv")
print(f"✓ Extratos carregados: {len(df_extra)} registros (ago/2025–jun/2026)")

# Consolidar
df_consolidado = pd.concat([
    df_prest[["mes_ano", "evento", "tipo", "valor", "macro_categoria"]],
    df_extra[["mes_ano", "evento", "tipo", "valor", "macro_categoria"]]
    # Extratos têm coluna 'complemento' mas prestações não, então dropamos
], ignore_index=True)

print(f"✓ Total consolidado: {len(df_consolidado)} registros")
print(f"  Período: {df_consolidado['mes_ano'].min()} → {df_consolidado['mes_ano'].max()}")

## 3. Extrair Lançamentos de Multa (REC.MULTA+C.M.+JRS.)

In [ ]:
print("\n" + "="*80)
print("SEÇÃO 2: Filtrar inadimplências (REC.MULTA+C.M.+JRS.)")
print("="*80)

# Filtrar eventos de multa (pode incluir variações)
multas_patterns = [
    "REC.MULTA+C.M.+JRS",
    "REC.MULTA",
    "REC. MULTA",
    "MULTA"
]

mask_multa = df_consolidado["evento"].str.upper().str.contains(
    "|".join(multas_patterns), na=False
)

df_multas = df_consolidado[mask_multa & (df_consolidado["tipo"] == "RECEITA")].copy()
print(f"✓ Lançamentos de multa encontrados: {len(df_multas)} registros")

if len(df_multas) == 0:
    print("❌ Nenhuma multa encontrada! Verificando eventos únicos...")
    print(df_consolidado[mask_multa]["evento"].unique()[:20])

## 4. Rastrear Apartamentos por Complemento

In [ ]:
print("\n" + "="*80)
print("SEÇÃO 3: Rastrear apartamentos (campo complemento)")
print("="*80)

# Para prestações, precisamos do campo complemento que pode estar em df_prest original
# Vamos carregar novamente com complemento se disponível
df_prest_full = pd.read_csv(CSV_DIR / "prestacoes.csv")
df_extra_full = pd.read_csv(CSV_DIR / "extratos.csv")

# Combinar com complemento quando disponível
df_prest_multas = df_prest_full[
    (df_prest_full["evento"].str.upper().str.contains("MULTA", na=False)) &
    (df_prest_full["tipo"] == "RECEITA")
].copy()

df_extra_multas = df_extra_full[
    (df_extra_full["evento"].str.upper().str.contains("MULTA", na=False)) &
    (df_extra_full["tipo"] == "RECEITA")
].copy()

print(f"✓ Multas prestações: {len(df_prest_multas)} registros")
print(f"✓ Multas extratos: {len(df_extra_multas)} registros")

# Consolidar multas com complemento (apartamento)
df_multas_completo = pd.concat([
    df_prest_multas[["mes_ano", "evento", "tipo", "valor"]].assign(complemento=""),
    df_extra_multas[["mes_ano", "evento", "tipo", "valor", "complemento"]]
], ignore_index=True)

# Extrair AP (apartamento) do complemento
df_multas_completo["apartamento"] = df_multas_completo["complemento"].str.extract(r"(AP\.\d+|AP \d+)")
df_multas_completo["apartamento"] = df_multas_completo["apartamento"].str.replace(r"AP[. ]+", "", regex=True)

print(f"\n✓ Total de multas com complemento: {len(df_multas_completo)}")
print(f"✓ Multas com apartamento identificado: {df_multas_completo['apartamento'].notna().sum()}")
print(f"✓ Apartamentos únicos: {df_multas_completo['apartamento'].nunique()}")

## 5. Ranking de Inadimplentes Recorrentes

In [ ]:
print("\n" + "="*80)
print("SEÇÃO 4: Ranking de inadimplentes recorrentes")
print("="*80)

# Agrupar por apartamento
inadimplentes = df_multas_completo[
    df_multas_completo["apartamento"].notna()
].groupby("apartamento").agg({
    "valor": ["sum", "count", "mean"],
    "mes_ano": lambda x: f"{x.min()} a {x.max()}"
}).round(2)

inadimplentes.columns = ["total_multas", "qtd_ocorrencias", "multa_media", "periodo"]
inadimplentes = inadimplentes.sort_values("qtd_ocorrencias", ascending=False).reset_index()

print(f"\n✓ Ranking de inadimplentes (por frequência):")
print(inadimplentes.head(20).to_string(index=False))

# Exportar
inadimplentes.to_csv(CSV_DIR / "inadimplentes_ranking.csv", index=False)
print(f"\n✓ Exportado: inadimplentes_ranking.csv")

## 6. Série Temporal de Inadimplência

In [ ]:
print("\n" + "="*80)
print("SEÇÃO 5: Série temporal de inadimplência")
print("="*80)

# Agregação por mês
df_multas_mes = df_multas_completo.groupby("mes_ano").agg({
    "valor": "sum",
    "apartamento": lambda x: x.notna().sum()  # qtd de apartamentos com multa neste mês
}).reset_index()
df_multas_mes.columns = ["mes_ano", "valor_multas", "qtd_apartamentos"]

# Calcular % de cobrança (multas vs receita total)
df_receitas_mes = df_consolidado[
    df_consolidado["tipo"] == "RECEITA"
].groupby("mes_ano")["valor"].sum().reset_index()
df_receitas_mes.columns = ["mes_ano", "receita_total"]

df_inadimplencia_mes = df_multas_mes.merge(df_receitas_mes, on="mes_ano")
df_inadimplencia_mes["pct_multa_vs_receita"] = (
    df_inadimplencia_mes["valor_multas"] / df_inadimplencia_mes["receita_total"] * 100
).round(2)

print(f"\n✓ Série temporal de inadimplência:")
print(df_inadimplencia_mes.to_string(index=False))

# Exportar
df_inadimplencia_mes.to_csv(CSV_DIR / "inadimplencia_por_mes.csv", index=False)
print(f"\n✓ Exportado: inadimplencia_por_mes.csv")

## 7. Visualizações

In [ ]:
print("\n" + "="*80)
print("SEÇÃO 6: Visualizações de inadimplência")
print("="*80)

# Gráfico 1: Série temporal de multas vs receita
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))

# Eixo 1: Multas vs Receita
ax1_twin = ax1.twinx()
ax1.bar(df_inadimplencia_mes["mes_ano"], df_inadimplencia_mes["valor_multas"], 
        label="Multas (R$)", alpha=0.7, color="#e74c3c")
ax1_twin.plot(df_inadimplencia_mes["mes_ano"], df_inadimplencia_mes["pct_multa_vs_receita"],
        label="% Multas vs Receita", color="#e67e22", marker="o", linewidth=2, markersize=6)
ax1.set_ylabel("Multas (R$)", fontsize=11)
ax1_twin.set_ylabel("% Multas vs Receita", fontsize=11)
ax1.set_title("Série Temporal: Multas vs Receita Total", fontsize=12, fontweight="bold")
ax1.grid(alpha=0.3)
ax1.legend(loc="upper left")
ax1_twin.legend(loc="upper right")

# Eixo 2: Apartamentos com multa
ax2.plot(df_inadimplencia_mes["mes_ano"], df_inadimplencia_mes["qtd_apartamentos"],
        label="Qtd de Apartamentos", color="#3498db", marker="s", linewidth=2, markersize=6)
ax2.fill_between(range(len(df_inadimplencia_mes)), df_inadimplencia_mes["qtd_apartamentos"], 
                  alpha=0.3, color="#3498db")
ax2.set_xlabel("Mês", fontsize=11)
ax2.set_ylabel("Qtd de Apartamentos com Multa", fontsize=11)
ax2.set_title("Incidência de Inadimplência (Qtd de Apartamentos)", fontsize=12, fontweight="bold")
ax2.grid(alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(FIG_DIR / "inadimplencia_serie_temporal.png", dpi=200, bbox_inches="tight")
print("✓ Gráfico 1 salvo: inadimplencia_serie_temporal.png")
plt.show()

In [ ]:
# Gráfico 2: Top 10 apartamentos com mais multas
top10 = inadimplentes.head(10).copy()

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(range(len(top10)), top10["qtd_ocorrencias"], color="#e74c3c", alpha=0.8)
ax.set_yticks(range(len(top10)))
ax.set_yticklabels([f"AP {ap}" for ap in top10["apartamento"]])
ax.set_xlabel("Qtd de Ocorrências", fontsize=11)
ax.set_title("Top 10 Apartamentos com Mais Multas (Inadimplentes Recorrentes)", 
             fontsize=12, fontweight="bold")
ax.grid(alpha=0.3, axis="x")

# Adicionar valor no topo de cada barra
for i, (idx, row) in enumerate(top10.iterrows()):
    ax.text(row["qtd_ocorrencias"] + 0.2, i, f"{int(row['qtd_ocorrencias'])}", 
            va="center", fontsize=10)

plt.tight_layout()
plt.savefig(FIG_DIR / "inadimplencia_top10.png", dpi=200, bbox_inches="tight")
print("✓ Gráfico 2 salvo: inadimplencia_top10.png")
plt.show()

In [ ]:
# Gráfico 3: Heatmap de inadimplência por mês e apartamento
# Criar matriz pivot
df_multas_pivot = df_multas_completo[
    df_multas_completo["apartamento"].notna()
].copy()

top_apts = inadimplentes.head(15)["apartamento"].tolist()  # Top 15 apartamentos
df_multas_pivot_filtered = df_multas_pivot[df_multas_pivot["apartamento"].isin(top_apts)]

pivot = pd.crosstab(
    index=df_multas_pivot_filtered["apartamento"],
    columns=df_multas_pivot_filtered["mes_ano"],
    values=df_multas_pivot_filtered["valor"],
    aggfunc="sum",
    margins=False
)

fig, ax = plt.subplots(figsize=(16, 8))
sns.heatmap(pivot, annot=True, fmt=".0f", cmap="YlOrRd", cbar_kws={"label": "Valor de Multa (R$)"}, ax=ax)
ax.set_title("Heatmap de Multas: Apartamentos × Meses", fontsize=12, fontweight="bold")
ax.set_xlabel("Mês", fontsize=11)
ax.set_ylabel("Apartamento", fontsize=11)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(FIG_DIR / "inadimplencia_heatmap.png", dpi=200, bbox_inches="tight")
print("✓ Gráfico 3 salvo: inadimplencia_heatmap.png")
plt.show()

## 8. Resumo Executivo

In [ ]:
print("\n" + "="*80)
print("RESUMO EXECUTIVO — ANÁLISE DE INADIMPLÊNCIA")
print("="*80)

# Estatísticas gerais
total_multas = df_inadimplencia_mes["valor_multas"].sum()
total_receita = df_inadimplencia_mes["receita_total"].sum()
pct_multa = (total_multas / total_receita * 100) if total_receita > 0 else 0

print(f"""
╔════════════════════════════════════════════════════════════════════════════════╗
║ MÉTRICAS CONSOLIDADAS (mai/2022–jun/2026)                                      ║
╠════════════════════════════════════════════════════════════════════════════════╣
│                                                                                ║
│  Total de Multas Recebidas:      R$ {total_multas:>15,.2f}      │
│  Total de Receitas:               R$ {total_receita:>15,.2f}      │
│  Multas % das Receitas:              {pct_multa:>16.2f}%      │
│                                                                                ║
│  Apartamentos com Multa:         {df_multas_completo['apartamento'].nunique():>16} unidades │
│  Ocorrências de Multa:           {len(df_multas_completo):>16} lançamentos │
│  Multa Média por Ocorrência:     R$ {df_multas_completo['valor'].mean():>15,.2f}      │
│                                                                                ║
│  Inadimplente Mais Recorrente:   AP {inadimplentes.iloc[0]['apartamento']:>15} ({int(inadimplentes.iloc[0]['qtd_ocorrencias'])}) │
│  Valor Máximo de Multa:          R$ {inadimplentes['total_multas'].max():>15,.2f}      │
│                                                                                ║
╚════════════════════════════════════════════════════════════════════════════════╝
""")

# Tendência
mes_inicial = df_inadimplencia_mes.iloc[0]
mes_final = df_inadimplencia_mes.iloc[-1]

print(f"""
╔════════════════════════════════════════════════════════════════════════════════╗
║ TENDÊNCIA: Primeiro Mês × Último Mês                                           ║
╠════════════════════════════════════════════════════════════════════════════════╣
│                                                                                ║
│  Período Inicial:  {mes_inicial['mes_ano']}                                           │
│    • Multas: R$ {mes_inicial['valor_multas']:>12,.2f}   |   Apartamentos: {int(mes_inicial['qtd_apartamentos']):>3}   │
│    • % vs Receita: {mes_inicial['pct_multa_vs_receita']:>5.2f}%                                         │
│                                                                                ║
│  Período Final:    {mes_final['mes_ano']}                                           │
│    • Multas: R$ {mes_final['valor_multas']:>12,.2f}   |   Apartamentos: {int(mes_final['qtd_apartamentos']):>3}   │
│    • % vs Receita: {mes_final['pct_multa_vs_receita']:>5.2f}%                                         │
│                                                                                ║
║  Variação: {((mes_final['valor_multas'] / mes_inicial['valor_multas'] - 1) * 100) if mes_inicial['valor_multas'] > 0 else 0:>+7.1f}%                                                    │
╚════════════════════════════════════════════════════════════════════════════════╝
""")

print("\n✓ Análise de inadimplência concluída!")
print(f"  Arquivos exportados: inadimplentes_ranking.csv, inadimplencia_por_mes.csv")
print(f"  Gráficos exportados: inadimplencia_serie_temporal.png, inadimplencia_top10.png, inadimplencia_heatmap.png")